# Neural Network Design & Hyperparameter Optimisation
## TensorFlow/Keras Grid Search on UCI Spambase

Binary classification neural network with systematic grid search over epoch and batch-size configurations on the UCI Spambase dataset (4,601 emails, 57 features), achieving 94.6% test accuracy.

**Author:** Raquel J. | [rjdatavoyage.co.uk](https://rjdatavoyage.co.uk)  
**Full case study:** [View on portfolio](https://rjdatavoyage.co.uk/projects/neural-network-architecture/)


In [ ]:
# Data loading and preparation

# URL to import data set from GitHub.
url = 'https://raw.githubusercontent.com/fourthrevlxd/cam_dsb/main/spamdata.csv'

In [ ]:
# 1. Importing the libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
tf.random.set_seed(42)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from tensorflow.keras.callbacks import EarlyStopping
import csv
import urllib.request
import os # import the os module to use operating system related functions

In [ ]:
# The data set
df = pd.read_csv(url, header=None)

In [ ]:
df.shape

(4601, 58)

In [ ]:
# 2. Checking the data set
print(df.head())

     0     1     2    3     4     5     6     7     8     9   ...    48  \
0  0.00  0.64  0.64  0.0  0.32  0.00  0.00  0.00  0.00  0.00  ...  0.00   
1  0.21  0.28  0.50  0.0  0.14  0.28  0.21  0.07  0.00  0.94  ...  0.00   
2  0.06  0.00  0.71  0.0  1.23  0.19  0.19  0.12  0.64  0.25  ...  0.01   
3  0.00  0.00  0.00  0.0  0.63  0.00  0.31  0.63  0.31  0.63  ...  0.00   
4  0.00  0.00  0.00  0.0  0.63  0.00  0.31  0.63  0.31  0.63  ...  0.00   

      49   50     51     52     53     54   55    56  57  
0  0.000  0.0  0.778  0.000  0.000  3.756   61   278   1  
1  0.132  0.0  0.372  0.180  0.048  5.114  101  1028   1  
2  0.143  0.0  0.276  0.184  0.010  9.821  485  2259   1  
3  0.137  0.0  0.137  0.000  0.000  3.537   40   191   1  
4  0.135  0.0  0.135  0.000  0.000  3.537   40   191   1  

[5 rows x 58 columns]


In [ ]:
# 3. Define features (X) and target (y)
# Here the the information gets separated (like words in emails)
# from what needs to be predicted (if it's spam or not).
# The last column in the data set indicates if each email is spam or not.

X = df.iloc[:, :-1]  # All columns except the last one
y = df.iloc[:, -1]   # Last column for classification

In [ ]:
# 4. Here the data is divided into three parts:
# Training Set: to teach the model how to classify emails
# Validation Set: For practice test to see how well the  model is learning without using the final check (test set).
# Test Set: This is the final exam where it's checked if the model can classify new emails it hasn't seen before.

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) #  42 is chosen to ensure that the random processes in the code are reproducible.

# Create validation set from training data
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

In [ ]:
# 5.1 Standardising the features
# making all the data features (like word counts) have the same scale.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 5.2 Defining the model

# model = Sequential([
#     Input(shape=(X_train.shape[1],)), # Using the Input layer to define the input shape
#     # the Input layer as the first layer in the Sequential model
#     Dense(64, activation='relu'),
#     Dense(32, activation='relu'),
#     Dense(1, activation='sigmoid')
#     ])

# Define the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Notes: The first hidden layer has 64 neurons with ReLU (Rectified Linear Unit)
# activation.

# The second hidden layer has 32 neurons, also with ReLU activation.
# Using a smaller number of neurons can help the network learn hierarchical
# representations and reduce the risk of overfitting.

# The output layer has a single neuron with sigmoid activation.
# Sigmoid is appropriate for binary classification tasks as it outputs a
# probability between 0 and 1, helping to classify the input as either spam or
# non-spam.

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Compile the model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy']) # loss='binary_crossentropy to measure how far off its predictions are from the true outcomes

In [ ]:
# Train the model
history = model.fit(X_train_scaled, y_train,
                    validation_data=(X_val_scaled, y_val),
                    epochs=10, # 10 epochs to avoid overfitting
                    batch_size=64, # 64 is a reasonable initial choice that balances computational efficiency with model performance
                    verbose=1)

Epoch 1/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.7225 - loss: 0.5600 - val_accuracy: 0.9076 - val_loss: 0.3255
Epoch 2/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9172 - loss: 0.2722 - val_accuracy: 0.9212 - val_loss: 0.2308
Epoch 3/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9356 - loss: 0.1855 - val_accuracy: 0.9266 - val_loss: 0.2279
Epoch 4/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9429 - loss: 0.1547 - val_accuracy: 0.9321 - val_loss: 0.2349
Epoch 5/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9560 - loss: 0.1314 - val_accuracy: 0.9293 - val_loss: 0.2349
Epoch 6/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9629 - loss: 0.1166 - val_accuracy: 0.9266 - val_loss: 0.2344
Epoch 7/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9659 - loss: 0.1042 - val_accuracy: 0.9293 - val_loss: 0.2438
Epoch 8/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9703 - loss: 0.0960 - val_accuracy: 0.9321 - val_loss

In [ ]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.2033
Test Accuracy: 0.9370


**The Test Loss increases and Test Accuracy starts dominishing with new retraining and re-evaluation attempts**

In [ ]:
# Create a vector of different batch_sizes=np.array([16, 32, 64]) and loop
# through it, retraining the model each time, and print the performances.
# Use the same model and number of epochs.

batch_sizes=np.array([16, 32, 64])
for batch_size in batch_sizes:
  model = Sequential([
    Dense(64, activation='relu', input_shape=(X.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
  ])

  model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
  history = model.fit(X_train_scaled, y_train,
                      validation_data=(X_val_scaled, y_val),
                      epochs=10,
                      batch_size=batch_size,
                      verbose=1)
  test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
  print(f"Batch Size: {batch_size}")
  print(f"Test Loss: {test_loss:.4f}")
  print(f"Test Accuracy: {test_accuracy:.4f}")
  print()

Epoch 1/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7427 - loss: 0.5049 - val_accuracy: 0.9239 - val_loss: 0.2246
Epoch 2/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9309 - loss: 0.1916 - val_accuracy: 0.9321 - val_loss: 0.2105
Epoch 3/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9438 - loss: 0.1546 - val_accuracy: 0.9321 - val_loss: 0.2054
Epoch 4/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9513 - loss: 0.1330 - val_accuracy: 0.9375 - val_loss: 0.1972
Epoch 5/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9624 - loss: 0.1183 - val_accuracy: 0.9321 - val_loss: 0.2051
Epoch 6/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9674 - loss: 0.1054 - val_accuracy: 0.9293 - val_loss: 0.2148
Epoch 7/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9682 - loss: 0.1028 - val_accuracy: 0.9239 - val_loss: 0.2175
Epoch 8/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9716 - loss: 0.0891 - val_accuracy: 0.

In [ ]:
# Create a vector of different epochs=np.array([10, 20, 30]) and loop through
# it, retraining the model each time and using the batch size.
# Jot down which model gave you the highest accuracy.

epochs=np.array([10, 20, 30])
for epoch in epochs:
  model = Sequential([
    Dense(64, activation='relu', input_shape=(X.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid'),
  ])
  # Compile the model again
  model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
  # Train the model
  history = model.fit(X_train_scaled, y_train,
                      validation_data=(X_val_scaled, y_val),
                      epochs=epoch,
                      batch_size=64,  # Use this for consistency with batch size experiments
                      verbose=1)
  # Evaluate the model
  test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
  print(f"Epochs: {epoch}")
  print(f"Test Loss: {test_loss:.4f}")
  print(f"Test Accuracy: {test_accuracy:.4f}")
  print()

Epoch 1/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.6099 - loss: 0.6302 - val_accuracy: 0.8587 - val_loss: 0.4138
Epoch 2/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8934 - loss: 0.3633 - val_accuracy: 0.9293 - val_loss: 0.2156
Epoch 3/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9356 - loss: 0.1986 - val_accuracy: 0.9375 - val_loss: 0.2054
Epoch 4/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9420 - loss: 0.1618 - val_accuracy: 0.9321 - val_loss: 0.2035
Epoch 5/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9481 - loss: 0.1408 - val_accuracy: 0.9321 - val_loss: 0.2013
Epoch 6/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9537 - loss: 0.1251 - val_accuracy: 0.9375 - val_loss: 0.2095
Epoch 7/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9587 - loss: 0.1134 - val_accuracy: 0.9348 - val_loss: 0.2114
Epoch 8/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9620 - loss: 0.1049 - val_accuracy: 0.9375 - val_loss

### **Best hyperparameters calculation using the Grid Search approach**


In [ ]:
# Hyperparameters for grid search
batch_sizes = np.array([16, 32, 64])
epochs = np.array([10, 20, 30])

# Lists to store results from multiple runs
best_accuracies = []
best_epochs_list = []
best_batch_sizes = []

for _ in range(100):  # Run 10 times
    best_accuracy = 0
    best_epochs = 0
    best_batch_size = 0

    model = Sequential([
      Dense(64, activation='relu', input_shape=(X.shape[1],)),
      Dense(32, activation='relu'),
      Dense(16, activation='relu'),
      Dense(16, activation='relu'),
      Dense(16, activation='relu'),
      Dense(16, activation='relu'),
      Dense(1, activation='sigmoid'),
    ])

    # Compile the model again
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

    # Early stopping callback to avoid overfitting
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    for batch_size in batch_sizes:
        for epoch in epochs:
            # Reset the model weights before training
            model = Sequential([
              Dense(64, activation='relu', input_shape=(X.shape[1],)),
              Dense(32, activation='relu'),
              Dense(16, activation='relu'),
              Dense(16, activation='relu'),
              Dense(16, activation='relu'),
              Dense(16, activation='relu'),
              Dense(1, activation='sigmoid'),
            ])
            model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

            # Train the model
            history = model.fit(X_train_scaled, y_train,
                                validation_data=(X_val_scaled, y_val),
                                epochs=epoch,
                                batch_size=batch_size,
                                callbacks=[early_stopping],
                                verbose=0)  # Setting verbose to 0 to reduce output for 10 runs
            # Evaluate the model
            test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

            # Record the best hyperparameters
            if test_accuracy > best_accuracy:
                best_accuracy = test_accuracy
                best_epochs = epoch
                best_batch_size = batch_size

    # Store the best parameters from this run
    best_accuracies.append(best_accuracy)
    best_epochs_list.append(best_epochs)
    best_batch_sizes.append(best_batch_size)

# Calculate averages
avg_best_accuracy = np.mean(best_accuracies)
avg_best_epochs = np.mean(best_epochs_list)
avg_best_batch_size = np.mean(best_batch_sizes)

# Print the average results
print(f"Average Best Accuracy: {avg_best_accuracy:.4f}")
print(f"Average Best Number of Epochs: {avg_best_epochs:.2f}")
print(f"Average Best Batch Size: {avg_best_batch_size:.2f}")

Average Best Accuracy: 0.9450
Average Best Number of Epochs: 14.50
Average Best Batch Size: 17.44


**Based on the results provided from the grid search (above):**

* Average Best Accuracy: 0.9450
* Average Best Number of Epochs: 14.50
* Average Best Batch Size: 17.44

**Number of Epochs:**

The average best number of epochs is 14.50. Since can't use half epochs, round this to either 14 or 15. **To avoid overfitting, will select 14 epochs as a conservative estimate**, but if opting to ensure the capture of the full learning potential, 15 epochs could be considered.

**Batch Size:**

The average best batch size is 17.44. Batch sizes are typically powers of 2 for computational efficiency, could round this to the nearest practical size. The closest power of 2 would be 16, but since 17 is closer to 32 than 16, and considering the nature of the results, could also opt for 32 as the batch size. For consistency with the grid search values tested (16, 32, 64), **16 would be the choice.**

**Recommended Hyperparameters:**
* Epochs: 14 or 15 (preferably 14 to prevent overfitting)
* Batch Size: 16 or 32 (preferably 16 for sticking closer to the average from the experiment)

**Rationale:**

These selections are based on the average performance across 100 runs, suggesting they should provide a good compromise between model accuracy and training efficiency.

The choice of 14 epochs is slightly conservative, aiming to stop training before overfitting might occur, while 16 for batch size aligns with the computational efficiency of powers of 2 and is very close to the average observed.


**Note**: The code below creates the results2.csv file in append mode and creates a CSV writer object. It then writes a new row to the CSV file containing the values of epoch, batch_size, test_loss, and test_accuracy.

In [ ]:
# URL for the CSV file
url = "https://colab.research.google.com/drive/"

try:
    # Fetch the CSV file from the server
    response = urllib.request.urlopen(url)
    data = response.read().decode("utf-8")  # Decode the response as UTF-8

    # Write the downloaded data to a local file, overwriting any existing content
    with open('results2.csv', 'w', newline='') as file:
        file.write(data)

    # Append new data to the file
    with open('results2.csv', 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([epoch, batch_size, test_loss, test_accuracy])

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
# To confirm the file was indeed created with content
if os.path.exists('results2.csv'):
    print("File 'results2.csv' exists and was created.")
else:
    print("File 'results2.csv' does not exist or wasn't created.")

File 'results2.csv' exists and was created.


In [ ]:
# Check if the file exists before attempting to open it
if os.path.exists('results2.csv'):
    os.system('start "" "results2.csv"')  # This works on Windows to open the file
    # For other operating systems, you might need different commands:
    # os.system('open results2.csv')  # macOS
    # os.system('xdg-open results2.csv')  # Linux
else:
    print("The file 'results2.csv' does not exist.")

In [ ]:
print(os.getcwd())

/content


# References

Hopkins, M., Reeber, E., Forman, G., Suermondt, J., 1999. Spambase. [online]. Available at: https://archive.ics.uci.edu/dataset/94. [Accessed 5 March 2024].